**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

# Exercise Sheet 09

## Recap

### First-Order Necessecary Conditions

The overall goal of this lecture series is to get a
feeling for solving the general nonlinear problem
$$
\begin{aligned}
&\min_{\mathbf w\in ℝ^q}
  L(\mathbf w)
      &\text{subject to}\\
&c_i(\mathbf w) = 0, &∀i\in \mathcal E,\\
&c_i(\mathbf w) ≤ 0, &∀i\in \mathcal I,
\end{aligned}
\tag{NL}
$$
where $\mathcal E\subset ℕ_0$ and $\mathcal I \subset ℕ_0$ index
the nonlinear equality and inequality constraints.
The so called **Lagrangian** $\mathcal L$ of this problem is
$$
\mathcal L(\mathbf w, \mathbf λ, \mathbf μ)
=
L(\mathbf w) + \sum_{i \in \mathcal I} λ_i c_i(\mathbf w) + \sum_{i \in \mathcal E} μ_i c_i(\mathbf w).
$$
Under certain conditions, the Lagrangian can be used to give
necessary conditions for a point $\mathbf w^*$ to be optimal for
(NL).
If $\mathbf w^*$ is locally optimal, then there are multipliers $\mathbf λ^*$
and $\mathbf μ^*$ such that
$$
\begin{aligned}
∇_{\mathbf w} \mathcal L(\mathbf w^*, \mathbf λ^*, \mathbf μ^*) &= \mathbf 0,  \\
c_i(\mathbf w^*) &= 0, & ∀i \in \mathcal E, \\
c_i(\mathbf w^*) &≤ 0, & ∀i \in \mathcal I, \\
λ_i &\geq 0, & ∀i\in \mathcal I,\\
λ_i c_i(\mathbf w^*) &= 0 & ∀i\in \mathcal I.\\
\end{aligned}
$$
These conditions are stated as Theorem 12.1 in “Numerical Optimization” by Nocedal,
and are referred to as the KKT conditions.
The last line, the complementary slackness condition,
tells us, that the multipliers of the inactive
inequality constraints ($c_i(\mathbf w^*) < 0$) are zero.
The first line can be re-written as
$$
\nabla_{\mathbf w}
\mathcal L(\mathbf w^*, \mathbf λ^*, \mathbf μ^*)
=
∇L(\mathbf w)
+
\sum_{i \in \mathcal A(\mathbf w^*)}
λ_i \nabla c_i(\mathbf w^*)
+
\sum_{i\in \mathcal E}
μ_i \nabla c_i(\mathbf w^*)
=
\mathbf 0,
$$
with the active constraint indices $\mathcal A (\mathbf w^*) = \{i \in \mathcal I: c_i(\mathbf w^*) = 0\}$.

## Active-Set Method for Convex QP

An important building block for solving (NL) with
**sequential quadratic programming** (SQP) is the
minimization of the convex quadratic problem
$$
\begin{aligned}
&\min_{\mathbf p \in ℝ^q}
  \frac{1}{2} \mathbf p^T \mathbf H \mathbf p
+
  \mathbf g^T \mathbf p,
&\text{subject to}\\
&\mathbf c_i^T \mathbf p + d_i = 0,  &∀i \in \mathcal E, \\
&\mathbf c_i^T \mathbf p + d_i ≤ 0,  &∀i \in \mathcal I,
\end{aligned}
\tag{QP}
$$
where $\mathbf H$ is a square $q\times q$ matrix and positive semi-definite,
and $\mathbf g$ is a vector in $ℝ^q$.
$\mathcal E\subset ℕ_0$ is the index set for equality constraints,
$\mathcal I\subset ℕ_0$ is the index set for inequality constraints,
and $\mathbf c_i, i\in \mathcal I\cup \mathcal E,$ are vectors in $ℝ^q$.

To solve (NL) using SQP methods, we take an initial guess $\mathbf w^{(0)}$ and solve problems of the form (QP) to
obtain a step $\mathbf p^{(\ell)}$, resulting in updates $\mathbf w^{(\ell+1)} = \mathbf w^{(\ell)} + \mathbf p^{(\ell)}$.
The matrix $\mathbf H$ is the Hessian of $\mathcal L(\mathbf w^{(\ell)})$ and $\mathbf g$ is the loss gradient at $\mathbf w^{(\ell)}$.

But for now, we focus on just solving (QP).
Best forget about the outer iteration indices $(\ell)$ …
Given a concrete instance of (QP), we want to find optimal $\mathbf p^*$ using the Active-Set method.
The Active-Set method is an iterative scheme in which we start with an initial guess
$\mathbf p^{(0)} \in ℝ^q$
and again solve a sequence of sub-problems to determine some step
$\mathbf s^{(k)} = \mathbf p^{(k+1)} - \mathbf p^{(k)}$.
Actually, we have a two-stage procedure.
First, we obtain $\tilde {\mathbf s}^{(k)}$ as the optimizer of
$$
\begin{aligned}
&\min_{\mathbf s \in ℝ^q}
\frac{1}{2} \mathbf s^T \mathbf H \mathbf s
+ \mathbf s^T \mathbf h^{(k)}
&\text{s.t.}\\
&\mathbf c_i^T \mathbf s = 0,  &∀i \in \mathcal E, \\
&\mathbf c_i^T \mathbf s = 0,  &∀i \in \mathcal A^{(k)}.
\end{aligned}
\tag{SP}
$$
Here, we have defined
$$
\mathbf h^{(k)} = \mathbf H \mathbf p^{(k)} + \mathbf g.
$$
The index set $\mathcal A^{(k)} \subset \mathcal I$ is the
iteration-dependent *working set* of active constraints at $\mathbf p^{(k)}$:
$$
\mathcal A^{(k)} = \left\{
   i \in \mathcal I: \mathbf c_i^T \mathbf p^{(k)} + d_i = 0
\right\}.
$$
In a second step, we compute a step-size (Exercise 1c) to obtain update step $\mathbf s^{(k)}$ from $\tilde{\mathbf s}^{(k)}$.

### Exercise 1a) 
In the cell below, complete the code to implement a function, that identifies the active
indices of the linear inequality constraints.
(That is, the inequality value is between 0 and `tol`.)
Suppose, the linear inequality constraints are given in matrix form by a matrix `C_ineq`
and `d_ineq`, such that
$$
\mathbf C_{\text{ineq}} \mathbf p^{(k)} + \mathbf d_{\text{ineq}} \stackrel{!}{≤} \mathbf 0,
$$
i.e., the rows of `C_ineq` hold the constraint vectors, and we are interested in
a subset of the row indices. `active_indices` should return a `Vector{Int}` holding these indices.

### Understanding the Concept

#### 1. What is an Active Constraint?

The problem defines a set of linear inequality constraints in the form:


$$\mathbf{C}_{\text{ineq}}\mathbf{p}^{(k)} + \mathbf{d}_{\text{ineq}} \le \mathbf{0}$$

Geometrically, each row in this matrix equation represents a boundary line (or hyperplane) in space.

* **Inactive:** If evaluating a row gives a negative number (e.g., $-5.2 \le 0$), the point $\mathbf{p}^{(k)}$ is safely inside the feasible region away from the boundary.
* **Violated (Infeasible):** If evaluating a row gives a positive number (e.g., $2.1 \not\le 0$), the point has crossed the boundary into forbidden territory. This is why the code throws an error if `val > tol`.
* **Active:** If evaluating a row gives exactly zero, the point is resting directly on that specific constraint boundary. We say the constraint is "active" or "binding."

#### 2. Why use a Tolerance (`tol`)?

In pure mathematics, an active constraint evaluates exactly to $0$. However, in computer science, floating-point arithmetic introduces tiny rounding errors. If a solver finds a point on a boundary, the computer's calculation might result in `1.2e-14` or `-3.5e-13` instead of a perfect `0.0`.

If you use `val == 0.0`, the code will almost always fail to find active constraints. The `tol` parameter acts as a tiny buffer zone. If the absolute value is smaller than this tolerance ($|v| \le \text{tol}$), we safely assume the algorithm intended to be exactly on the boundary.

#### 3. How this fits into Optimization

In algorithms like the Active Set Method, the solver needs to know exactly which walls it is bumping into. By returning the `active_idx` array, the algorithm can isolate only the relevant rows from $\mathbf{C}_{\text{ineq}}$ to calculate the necessary Lagrange multipliers ($\lambda$) for the complementary slackness KKT condition.

In [1]:
import numpy as np

def active_indices(C_ineq, d_ineq, pk, tol=1e-10):
    """
    Identifies the active indices of linear inequality constraints.
    
    Parameters:
    C_ineq (np.ndarray): The constraint matrix.
    d_ineq (np.ndarray): The constraint vector.
    pk (np.ndarray): The current point/variable vector.
    tol (float): Numerical tolerance for floating-point comparisons.
    
    Returns:
    list: A list of integer indices representing the active constraints.
    """
    # Check dimensions (similar to @assert in Julia)
    assert C_ineq.shape[0] == len(d_ineq), "Row count of C_ineq must match length of d_ineq"
    
    # 1) Initialize empty list to hold active indices
    active_idx = []
    
    # 2) Evaluate constraint equation C * p + d
    # Using the @ operator for matrix multiplication in numpy
    constraint_values = C_ineq @ pk + d_ineq
    
    # 3) Iterate over result vector entries
    for i, val in enumerate(constraint_values):
        # a) if a value is > tol, throw an error
        if val > tol:
            raise ValueError(f"pk is not feasible at index {i}. Value {val} is greater than tolerance {tol}.")
            
        # b) if absolute value of entry is <= tol, add index to index array
        elif abs(val) <= tol:
            active_idx.append(i)
            
    # 4) return index array
    return active_idx

When checking `active_indices`, we can use the `Test` library to test for exceptions:

In [2]:
# Define the constraint matrices
# w1 <= 0.5*w2  =>  1*w1 - 0.5*w2 <= 0
# w1 <= 1       =>  1*w1 + 0*w2 - 1 <= 0
C_ineq = np.array([
    [1.0, -0.5],
    [1.0, 0.0]
])
d_ineq = np.array([0.0, -1.0])

# --- Test 1 ---
pk = np.array([0.5, 1.0])
Ak = active_indices(C_ineq, d_ineq, pk)

# Equivalent to @assert Ak isa AbstractVector{<:Integer}
assert isinstance(Ak, list), "Ak should be a list"

# Equivalent to @assert only(Ak) == 1
# In Python, we check length and the 0th index
assert len(Ak) == 1 and Ak[0] == 0, f"Expected only index 0, got {Ak}"


# --- Test 2 ---
pk = np.array([2.0, 1.0])

# Equivalent to Test.@test_throws Exception ...
exception_thrown = False
try:
    active_indices(C_ineq, d_ineq, pk)
except ValueError:
    exception_thrown = True
assert exception_thrown, "Expected a ValueError for an infeasible pk"


# ### BEGIN TESTS ###

# --- Test 3 ---
pk = np.array([1.0, 2.0])
Ak = active_indices(C_ineq, d_ineq, pk)

# Equivalent to @assert Ak == [1, 2]
assert Ak == [0, 1], f"Expected active indices [0, 1], got {Ak}"


# --- Test 4 ---
pk = np.array([1.0 - 1e-11, 2.0])
Ak = active_indices(C_ineq, d_ineq, pk, tol=1e-10)

# Equivalent to @assert Ak == [1, 2]
assert Ak == [0, 1], f"Expected active indices [0, 1] with tol=1e-10, got {Ak}"


# --- Test 5 ---
Ak = active_indices(C_ineq, d_ineq, pk, tol=1e-11)

# Equivalent to @assert isempty(Ak)
assert len(Ak) == 0, f"Expected an empty list with tol=1e-11, got {Ak}"

# ### END TESTS ###

print("All tests passed successfully!")

All tests passed successfully!


### Exercise 1b) 
In the cell below, complete the code to implement a function, that solves the KKT
equations for the subproblem (SP),
$$
\begin{bmatrix}
\mathbf H & \mathbf C_{\mathcal A^{(k)}}^T & \mathbf C_{\text{eq}}^T \\
\mathbf C_{\mathcal A^{(k)}}& \mathbf 0 & \mathbf 0 \\
\mathbf C_{\text{eq}} & \mathbf 0 & \mathbf 0
\end{bmatrix}
\begin{bmatrix}
\tilde{\mathbf s}^{(k)} \\
\mathbf λ^{(k)} \\
\mathbf μ^{(k)} \\
\end{bmatrix}
\stackrel{!}{=}
\begin{bmatrix}
-\mathbf h^{(k)} \\
\mathbf 0\\
\mathbf 0
\end{bmatrix}.
$$
These equations result from writing down the KKT conditions for (SP).

### Understanding the Concepts

The below code represents the **computational engine** of the algorithm we were looking at previously.

It solves a **KKT System** (also known as a saddle-point system), which is derived directly from the KKT conditions (Image 1).

#### 1. What is this matrix equation?

To find the next step $\tilde{\mathbf{s}}^{(k)}$ in an optimization algorithm like Sequential Quadratic Programming (SQP) or an Active Set method, the solver creates a simplified quadratic model of the objective function and a linear model of the constraints.

Setting the derivative of the Lagrangian of this subproblem to zero (the **Stationarity** condition) yields this exact linear system:

$$\begin{bmatrix} \mathbf{H} & \mathbf{C}_{\mathcal{A}^{(k)}}^T & \mathbf{C}_{\text{eq}}^T \\ \mathbf{C}_{\mathcal{A}^{(k)}} & \mathbf{0} & \mathbf{0} \\ \mathbf{C}_{\text{eq}} & \mathbf{0} & \mathbf{0} \end{bmatrix} \begin{bmatrix} \tilde{\mathbf{s}}^{(k)} \\ \lambda^{(k)} \\ \mu^{(k)} \end{bmatrix} = \begin{bmatrix} -\mathbf{h}^{(k)} \\ \mathbf{0} \\ \mathbf{0} \end{bmatrix}$$

#### 2. The Inputs (The Block Matrix and RHS)

* $\mathbf{H}$ **(Hessian Matrix):** Represents the curvature (second derivatives) of the objective function. It shapes the quadratic "bowl" the solver is trying to minimize.
* $\mathbf{C}_{\mathcal{A}^{(k)}}$ **and** $\mathbf{C}_{\text{eq}}$ **(Jacobians):** These are the gradients of the constraints. Because they are on the bottom rows multiplied by the step $\tilde{\mathbf{s}}^{(k)}$, they enforce the condition that the step must not violate the equality constraints or the *active* inequality constraints (it forces $\mathbf{C} \cdot \tilde{\mathbf{s}}^{(k)} = \mathbf{0}$, keeping the step strictly on the boundary).
* $-\mathbf{h}^{(k)}$ **(Negative Gradient):** Points in the direction of steepest descent.

#### 3. The Outputs (The Solution Vector)

By solving this one giant linear system, the algorithm gets three things simultaneously:

* $\tilde{\mathbf{s}}^{(k)}$ **(The Step Direction):** The optimal vector to move the variables to reduce the objective function while sliding exactly along the active constraint boundaries.
* $\lambda^{(k)}$ **and** $\mu^{(k)}$ **(The Lagrange Multipliers):** These represent the "forces" required to keep the solution from breaking through the constraints.
* If you recall the flowchart, if an element of $\lambda^{(k)}$ is negative, it means the constraint is actually pushing the solution *away* from the true minimum, signaling the algorithm to drop it from the active set in the next iteration.

In [3]:
def solve_step_problem(H, C_Ak, C_eq, hk):
    """
    Solves the KKT linear system for an optimization subproblem.
    
    Parameters:
    H (np.ndarray): Hessian matrix (size: num_vars x num_vars)
    C_Ak (np.ndarray): Active inequality constraints Jacobian (size: num_ineq x num_vars)
    C_eq (np.ndarray): Equality constraints Jacobian (size: num_eq x num_vars)
    hk (np.ndarray): Gradient vector (size: num_vars)
    
    Returns:
    tuple: (sk_tilde, lambdak, muk)
    """
    # Determine dimensions
    num_vars = H.shape[0]
    num_ineq = C_Ak.shape[0] if C_Ak is not None and C_Ak.size > 0 else 0
    num_eq = C_eq.shape[0] if C_eq is not None and C_eq.size > 0 else 0
    
    total_size = num_vars + num_ineq + num_eq
    
    # 1) Initialize KKT matrix and RHS vector with zeros
    KKT = np.zeros((total_size, total_size))
    RHS = np.zeros(total_size)
    
    # 2) Fill the Right-Hand Side (RHS)
    # The first 'num_vars' entries are -hk; the rest remain 0
    RHS[0:num_vars] = -hk
    
    # 3) Assemble the KKT Block Matrix
    # Top-Left Block: H
    KKT[0:num_vars, 0:num_vars] = H
    
    if num_ineq > 0:
        # Mid-Left Block: C_Ak
        KKT[num_vars : num_vars+num_ineq, 0:num_vars] = C_Ak
        # Top-Mid Block: C_Ak^T
        KKT[0:num_vars, num_vars : num_vars+num_ineq] = C_Ak.T
        
    if num_eq > 0:
        # Bottom-Left Block: C_eq
        KKT[num_vars+num_ineq : total_size, 0:num_vars] = C_eq
        # Top-Right Block: C_eq^T
        KKT[0:num_vars, num_vars+num_ineq : total_size] = C_eq.T
        
    # 4) Solve the linear equation system
    solution = np.linalg.solve(KKT, RHS)
    
    # 5) Extract the components
    sk_tilde = solution[0:num_vars]
    lambdak = solution[num_vars : num_vars+num_ineq]
    muk = solution[num_vars+num_ineq : total_size]
    
    return sk_tilde, lambdak, muk

Below are the tests for `solve_step_problem`. We use `LinearAlgebra` to initialize `H` as the identity matrix.

In [4]:
# --- Test Case 1 Setup ---
H = 1.0 * np.eye(2) # 2x2 because we have 2 variables (w1 and w2)
# Using 2D arrays to maintain matrix shapes (1 row, 2 columns)
C_Ak = np.array([[1.0, -0.5]])
C_eq = np.array([[1.0, 0.0]])
pk = np.array([1.0, 2.0])

# Calculate gradient (matrix multiplication)
hk = H @ pk

# Solve the subproblem
sk_tilde, lambda_k, mu_k = solve_step_problem(H, C_Ak, C_eq, hk)

# Type and shape assertions
assert isinstance(sk_tilde, np.ndarray), "sk_tilde must be a NumPy array"
assert isinstance(lambda_k, np.ndarray), "lambda_k must be a NumPy array"
assert isinstance(mu_k, np.ndarray), "mu_k must be a NumPy array"

assert len(sk_tilde) == 2
assert len(lambda_k) == 1
assert len(mu_k) == 1

# `sk_tilde` should be close to zero:
assert np.sum(sk_tilde ** 2) <= 1e-10

# ### BEGIN TESTS ###
# only(λk) ≈ 4 and only(μk) ≈ -5
assert np.isclose(lambda_k[0], 4.0), f"Expected lambda_k ≈ 4, got {lambda_k[0]}"
assert np.isclose(mu_k[0], -5.0), f"Expected mu_k ≈ -5, got {mu_k[0]}"


# --- Test Case 2 Setup (Empty Active Constraints) ---
# Create an empty 0x2 matrix to simulate no active inequality constraints
C_Ak = np.zeros((0, 2))
pk = np.array([1.0, 4.0])
hk = H @ pk

sk_tilde, lambda_k, mu_k = solve_step_problem(H, C_Ak, C_eq, hk)

# sk_tilde ≈ [0, -4]
assert np.allclose(sk_tilde, [0.0, -4.0]), f"Expected sk_tilde ≈ [0, -4], got {sk_tilde}"

# isempty(λk)
assert len(lambda_k) == 0, f"Expected lambda_k to be empty, got {lambda_k}"

# μk ≈ [-1]
assert np.allclose(mu_k, [-1.0]), f"Expected mu_k ≈ [-1], got {mu_k}"
# ### END TESTS ###

print("All KKT subproblem tests passed successfully!")

All KKT subproblem tests passed successfully!


### Exercise 1c)
Once $\tilde{\mathbf s}^{(k)}$ is available from `solve_step_problem`,
we need to determine a step size $α_k \in [0, 1]$.
$α_k$ should be as large as possible, and chosen so
that $\mathbf p^{(k+1)} = \mathbf p^{(k)} + α_k \tilde{\mathbf s}^{(k)}$ is
feasible for **all** the linear constraints of (QP).
The vector $\mathbf p^{(k+1)}$ is feasible for any $α_k$ and
all the constraints indexed by $\mathcal E$
and $\mathcal A^{(k)}$ by virtue of the KKT conditions of (SP).
Suppose $i\in \mathcal I\smallsetminus A^{(k)}$.
If $\mathbf c_i^T \tilde{\mathbf s}^{(k)} \leq 0$, then for all $α_k \geq 0$
we have
$$
\mathbf c_i^T(\mathbf p^{(k)} + α_k \tilde{\mathbf s}^{(k)}) + d_i
\leq
\mathbf c_i^T \mathbf p^{(k)} + d_i
\leq 0.
$$
Thus, we only have to consider the case
$i \in \mathcal I\smallsetminus \mathcal A^{(k)}$
and $\mathbf c_i^T \tilde{\mathbf s}^{(k)} > 0$.
In that case, we solve
$$
α_k
\leq
\frac{-d_i - \mathbf c_i^T \mathbf p^{(k)}}{\mathbf c_i^T \tilde{\mathbf s}^{(k)}}.
$$
Implement `stepsize_and_blocking_index`
to return the stepsize $α_k$ and a blocking index.
Use this formula:
$$
α_k = \min
\left\{
1,
\min_{
    \substack{
        i \in \mathcal I\smallsetminus \mathcal A^{(k)},\\
        \mathbf c_i^T \tilde{\mathbf s}^{(k)} > 0
    }
}
\frac{-d_i - \mathbf c_i^T \mathbf p^{(k)}}{\mathbf c_i^T \tilde{\mathbf s}^{(k)}}
\right\}.
$$
A "blocking index" is an index $i\in\mathcal I\smallsetminus \mathcal A^{(k)}$ such that the expression
of the inner $\min$ operator is strictly smaller than $1$.
If there is no blocking index, return the stepsize and $-1$.

### Understanding the Concepts

This exercise represents the **Line Search** phase of the Active Set algorithm.

In the previous step, the KKT system provided a step direction, $\tilde{\mathbf{s}}^{(k)}$. This direction is guaranteed to decrease your objective function while mathematically "sliding" perfectly along the constraints that are currently active.

However, you now need to determine **how far** to walk in that direction. The step size is a scalar value denoted as $\alpha_k$, bounded between $0$ and $1$. The new position will be:


$$\mathbf{p}^{(k+1)} = \mathbf{p}^{(k)} + \alpha_k \tilde{\mathbf{s}}^{(k)}$$

Ideally, you want to take the full step ($\alpha_k = 1$). But taking that full step might cause you to crash into a different constraint wall that wasn't previously active. In applications like Model Predictive Control (MPC) for autonomous systems, this is the exact mechanism that ensures the calculated control inputs for the next time step (like steering angle or acceleration) do not violate hard physical limits or track boundaries.

Here is the breakdown of the logic:

**1. Ignoring Active Constraints**
We don't need to check the constraints in the active set $\mathcal{A}^{(k)}$ or the equality constraints $\mathcal{E}$. The KKT system already inherently satisfied them. If you walk along the wall, you won't crash into the wall. We only care about the inactive constraints: $i \in \mathcal{I} \setminus \mathcal{A}^{(k)}$.

**2. The Direction Check ($\mathbf{c}_i^T \tilde{\mathbf{s}}^{(k)} > 0$)**
For every inactive constraint, we check the dot product of the constraint gradient ($\mathbf{c}_i^T$) and our step direction ($\tilde{\mathbf{s}}^{(k)}$).

* If $\mathbf{c}_i^T \tilde{\mathbf{s}}^{(k)} \le 0$, it means our step is carrying us parallel to the boundary or safely away from it. We can step as far as we want.
* If $\mathbf{c}_i^T \tilde{\mathbf{s}}^{(k)} > 0$, it means we are walking directly towards the boundary wall. We are on a collision course.

**3. Calculating the Collision Point**
For the constraints we are moving towards, we calculate exactly what fractional step size $\alpha$ will place us directly on the boundary ($0$). This is algebraic rearrangement of the constraint equation:


$$\alpha_k \le \frac{-d_i - \mathbf{c}_i^T \mathbf{p}^{(k)}}{\mathbf{c}_i^T \tilde{\mathbf{s}}^{(k)}}$$

**4. The Blocking Index**
We evaluate this fraction for all constraints we are moving towards. The constraint that yields the *smallest* $\alpha$ is the wall we will hit first.

* If this smallest $\alpha$ is less than $1$, it becomes our new step size, and the index of that constraint becomes the **blocking index** ($j$). This specific constraint will be added to the active set $\mathcal{A}$ in the next iteration of the algorithm.
* If the smallest $\alpha$ is $\ge 1$, we can safely take the full Newton step ($\alpha_k = 1$), and there is no blocking index ($j = -1$).

In [5]:
import numpy as np

def stepsize_and_blocking_index(C_ineq, d_ineq, pk, sk_tilde, Ak):
    """
    Determines the maximum allowable step size and the blocking constraint index.
    
    Parameters:
    C_ineq (np.ndarray): The inequality constraint matrix.
    d_ineq (np.ndarray): The inequality constraint offsets.
    pk (np.ndarray): The current variable vector.
    sk_tilde (np.ndarray): The calculated step direction.
    Ak (list or set): Indices of currently active inequality constraints.
    
    Returns:
    tuple: (alphak, j) where alphak is the step size and j is the blocking index.
    """
    # Initialize to a full step and no blocking index
    alphak = 1.0
    j = -1
    
    num_ineq = C_ineq.shape[0]
    
    # Iterate through all inequality constraints
    for i in range(num_ineq):
        # 1) Only evaluate inactive constraints
        if i not in Ak:
            c_i = C_ineq[i, :]
            
            # Dot product to check if we are moving towards the boundary
            c_s_dot = np.dot(c_i, sk_tilde)
            
            # 2) We only risk violation if c_s_dot > 0
            if c_s_dot > 0:
                
                # 3) Calculate the exact step size to hit the boundary
                numerator = -d_ineq[i] - np.dot(c_i, pk)
                alpha_temp = numerator / c_s_dot
                
                # 4) If this boundary is hit sooner than previously found ones, update
                if alpha_temp < alphak:
                    alphak = alpha_temp
                    j = i
                    
    return alphak, j

Let's test our function:

In [6]:
# --- Test Case 1 ---
# C_ineq = [1 -0.5]
C_ineq = np.array([[1.0, -0.5]])
# d_ineq = [0,]
d_ineq = np.array([0.0])

pk = np.array([1.0, 2.0])
sk_tilde = np.array([0.0, -2.0])
Ak = [] # Int[]

# Check if the returned values make sense:
alphak, j = stepsize_and_blocking_index(C_ineq, d_ineq, pk, sk_tilde, Ak)

assert isinstance(alphak, (float, np.floating)), "alphak must be a real number"
assert isinstance(j, (int, np.integer)), "j must be an integer"
assert -1e-10 <= alphak <= 1.0 + 1e-10

# ### BEGIN TESTS ###
assert abs(alphak) <= 1e-10, f"Expected alphak <= 1e-10, got {alphak}"
# Julia's index 1 is Python's index 0
assert j == 0, f"Expected j == 0, got {j}"
# ### END TESTS ###


# --- Test Case 2 ---
# Now, we should be able to take a full step:
Ak = [0] # Julia was [1,]
sk = np.zeros(2) # Note: The original Julia code declares `sk` here but uses `sk_tilde` in the function call below

alphak, j = stepsize_and_blocking_index(C_ineq, d_ineq, pk, sk_tilde, Ak)

assert np.isclose(alphak, 1.0), f"Expected alphak ≈ 1, got {alphak}"
assert j == -1, f"Expected j == -1, got {j}"
# ### BEGIN TESTS ###


# --- Test Case 3 ---
# C_ineq = [1 -0.5; 10 100]
C_ineq = np.array([
    [1.0, -0.5],
    [10.0, 100.0]
])
# d_ineq = [0, 1000]
d_ineq = np.array([0.0, 1000.0])
Ak = [] # Int[]
pk = np.array([1.0, 4.0])
sk_tilde = np.array([0, -4.0])

alphak, j = stepsize_and_blocking_index(C_ineq, d_ineq, pk, sk_tilde, Ak)

assert np.isclose(alphak, 0.5), f"Expected alphak ≈ 0.5, got {alphak}"
# Julia's index 1 is Python's index 0
assert j == 0, f"Expected j == 0, got {j}"
# ### END TESTS ###

print("All stepsize and blocking index tests passed successfully!")

All stepsize and blocking index tests passed successfully!


### Exercise 1d) 
We now have all building blocks for the Active-Set Method.
The algorithm stops, if a step $\mathbf s^{(k)}$ is zero
and all multipliers $λ_i^{(k)}$ are non-negative.
Otherwise, the working set is augmented.
Likewise, the working set is augmented in case of a non-zero
step with $α_k < 1$.

This final exercise brings together all the modular components we've built (1a, 1b, and 1c) to construct the main loop of the **Active-Set Method** for Quadratic Programming (QP). In the context of implementing controllers like MPC for autonomous lane changes, this solver is the computational core that runs at every time step to find the optimal control inputs while strictly adhering to system constraints.

### The Concepts: The Active-Set Main Loop

The algorithm iteratively explores the boundaries of the feasible region. At each iteration, it holds a specific set of constraints "active" (meaning it treats them temporarily as strict equalities) and tries to minimize the objective function along those boundaries.

The loop relies on a continuous cycle of evaluating KKT conditions and making routing decisions based on two primary branches:

**1. The "Zero Step" Branch (We hit a local minimum for the current boundaries)**
When `sk_tilde` is essentially zero, the algorithm has reached the absolute minimum of the objective function *given the current active walls*.

* **The Dual Feasibility Check:** It then examines the Lagrange multipliers ($\lambda$) for those active walls.
* **Optimal:** If all $\lambda \ge 0$, it means every active wall is actively pushing the solution "inward" toward the feasible region. The KKT conditions are fully satisfied, and the global optimum has been found.
* **Sub-optimal:** If a $\lambda < 0$, it means that specific wall is artificially preventing the objective function from decreasing further. The algorithm identifies the most negative multiplier, drops the corresponding constraint from the active set (`Ak`), and loops again to step away from that wall into deeper feasible space.

**2. The "Non-Zero Step" Branch (We are moving towards a minimum)**
If a step direction exists, the algorithm must take it. However, it uses the line search (`stepsize_and_blocking_index`) to ensure it doesn't accidentally violate an inactive constraint while moving.

* **No Blocking:** If the full step ($\alpha_k = 1$) is safe, it updates the state and loops.
* **Blocking:** If the step intersects a new boundary before reaching its target ($\alpha_k < 1$), it stops exactly on that boundary, adds the newly struck wall's index to the active set (`Ak`), and loops to recalculate a new direction that slides along this newly added wall.

Complete the marked sections in the code below:

In [7]:
import logging

# Configure basic logging to replicate Julia's @info
logging.basicConfig(level=logging.INFO, format='%(message)s')

def solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0, tol=1e-10, max_iter=100):
    """
    Solves a strictly convex Quadratic Program using the Active-Set Method.
    """
    pk = np.copy(p0)
    
    # 1) Check feasibility of p0 with respect to equality constraints
    if C_eq is not None and C_eq.size > 0:
        r_eq = C_eq @ pk + d_eq
        if not np.all(np.abs(r_eq) <= tol):
            raise ValueError("Initial point p0 is not feasible with respect to equality constraints.")

    # 2) Determine initial active indices for `pk`
    Ak = active_indices(C_ineq, d_ineq, pk, tol=tol)
    
    k = 1
    while k <= max_iter:
        # Compute the current objective value `Lk` of (QP) at `pk`
        # L = 0.5 * p^T * H * p + g^T * p
        Lk = 0.5 * np.dot(pk.T, H @ pk) + np.dot(g.T, pk)
        
        logging.info(f"Iteration {k}, L(pk)={Lk}")
        
        # Extract active inequality constraint matrix
        # In Python, if Ak is empty, this safely creates a 0-row array
        C_Ak = C_ineq[Ak, :] if len(Ak) > 0 else np.zeros((0, pk.shape[0]))
        
        # Solve subproblem to get step
        hk = H @ pk + g
        sk_tilde, lambda_k, mu_k = solve_step_problem(H, C_Ak, C_eq, hk)
        
        # Check if the step is practically zero
        if np.linalg.norm(sk_tilde) <= tol:
            logging.info("Zero step.")
            
            # If step is zero, check if all multipliers are non-negative
            problem_solved = len(lambda_k) == 0
            
            if not problem_solved:
                # Find the minimum multiplier and its index in the lambda_k array
                lambda_min_index = np.argmin(lambda_k)
                lambda_min = lambda_k[lambda_min_index]
                
                problem_solved = lambda_min >= -tol
                
            if problem_solved:
                logging.info("Problem solved.")
                break
                
            # If there are negative multipliers, remove the constraint corresponding 
            # to the most negative multiplier from the active set for the next iteration.
            # (pop removes the element at the specified index)
            Ak.pop(lambda_min_index)
            
        else:
            # sk_tilde is non-zero, compute stepsize and blocking index
            alphak, j = stepsize_and_blocking_index(C_ineq, d_ineq, pk, sk_tilde, Ak)
            
            # If there is a blocking index, add it to `Ak`
            if j != -1:
                Ak.append(j)
                
            # Update `pk` for the next iteration
            pk = pk + alphak * sk_tilde
            
        k += 1
        
    return pk, k

Here is how we can test your implementation:

In [8]:
# --- Setup Test Inputs ---
H = np.eye(2)
g = np.zeros(2)

# Matrices keep 2D structure (1 row, 2 columns)
C_ineq = np.array([[1.0, -0.5]])
d_ineq = np.array([0.0])

C_eq = np.array([[1.0, 0.0]])
d_eq = np.array([-1.0])

p0 = np.array([1.0, 4.0])

# --- Run the Active-Set Solver ---
# max_iter is set to 10 as shown in the image
popt, k = solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0, max_iter=10)

# --- Assertions & Verification ---
assert isinstance(popt, np.ndarray), "popt must be a NumPy array"
assert len(popt) == 2, f"Expected popt length to be 2, got {len(popt)}"
assert k == 2, f"Expected the solver to finish in 2 iterations, got {k}"

# Check that the solution satisfies constraints
# Translated from popt[1] -> popt[0] and popt[2] -> popt[1]
assert abs(popt[0] - 1.0) <= 1e-9, f"Equality constraint violation: {popt[0]}"
assert abs(popt[0] - 0.5 * popt[1]) <= 1e-9, f"Active inequality constraint violation"

# Check if the objective value was successfully reduced
def L(p):
    return 0.5 * np.dot(p.T, H @ p) + np.dot(g.T, p)

L0 = L(p0)
Lopt = L(popt)

assert Lopt < L0, f"Optimization failed to minimize objective. L0: {L0}, Lopt: {Lopt}"

print(f"All integration tests passed successfully!")
print(f"Optimal solution vector: {popt}")

Iteration 1, L(pk)=8.5
Iteration 2, L(pk)=2.5
Zero step.
Problem solved.


All integration tests passed successfully!
Optimal solution vector: [1. 2.]


In [9]:
import numpy as np
import logging

# NBGrader does not like logging, so we disable it sometimes:
logger = logging.getLogger()
original_level = logger.getEffectiveLevel()
logger.setLevel(logging.CRITICAL)

try:
    # --- Setup Problem Matrices ---
    H = 1.0 * np.eye(2)
    g = np.zeros(2)

    C_ineq = np.array([[1.0, -0.5]])
    d_ineq = np.array([0.0])

    C_eq = np.array([[1.0, 0.0]])
    d_eq = np.array([-1.0])

    # --- First Test Block ---
    p0 = np.array([1.0, 4.0])
    popt, k = solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0, max_iter=10)

    assert isinstance(popt, np.ndarray), "popt must be an array"
    assert len(popt) == 2
    assert k == 2

    # check that solution satisfies constraints (0-based indexing)
    assert abs(popt[0] - 1.0) <= 1e-9
    assert abs(popt[0] - 0.5 * popt[1]) <= 1e-9

    # check, if optimal value is reduced
    def L(p):
        return 0.5 * np.dot(p.T, H @ p) + np.dot(g.T, p)

    L0 = L(p0)
    Lopt = L(popt)
    assert Lopt < L0

    # ### BEGIN TESTS
    # --- Second Test Block ---
    p0 = np.array([1.0, 2.0])
    popt, k = solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0, max_iter=10)

    assert np.allclose(popt, [1.0, 2.0]), f"Expected popt ≈ [1, 2], got {popt}"
    assert k == 1, f"Expected k == 1, got {k}"
    # ### END TESTS

finally:
    # Restore logging back to its original state
    logger.setLevel(original_level)

print("All NBGrader test blocks passed successfully!")

All NBGrader test blocks passed successfully!


### Solving Analytically

### The Problem Definition

We are solving the following continuous optimization problem:

**Minimize:** 

$$L(\mathbf{p}) = \frac{1}{2}\mathbf{p}^T \mathbf{H} \mathbf{p} + \mathbf{g}^T\mathbf{p} \implies \frac{1}{2}(p_1^2 + p_2^2)$$

**Subject to:**

* **Equality Constraint ($\mathcal{E}$):** $\mathbf{C}_{eq}\mathbf{p} + \mathbf{d}_{eq} = \mathbf{0} \implies p_1 - 1 = 0 \implies p_1 = 1$
* **Inequality Constraint ($\mathcal{I}$):** $\mathbf{C}_{ineq}\mathbf{p} + \mathbf{d}_{ineq} \le \mathbf{0} \implies p_1 - 0.5p_2 \le 0$

Inital guess: $\mathbf{p}^{(0)} = \begin{bmatrix} 1 \\ 4 \end{bmatrix}$.

---

### Initialization

* **Current Point:** $\mathbf{p}^{(0)} = \begin{bmatrix} 1 \\ 4 \end{bmatrix}$
* **Equality Check:** $1 - 1 = 0$. (The point is feasible with respect to equalities).
* **Inequality Check:** Evaluate $p_1 - 0.5p_2 \implies 1 - 0.5(4) = -1$.
* Since $-1 < 0$, the point is strictly inside the feasible region for this constraint.
* **Active Set:** $\mathcal{A}^{(0)} = \emptyset$ (No inequality constraints are active).

---

### Iteration 1: Stepping towards the boundary

**1. Calculate the Gradient:**


$$\mathbf{h}^{(0)} = \mathbf{H}\mathbf{p}^{(0)} + \mathbf{g} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} 1 \\ 4 \end{bmatrix} + \begin{bmatrix} 0 \\ 0 \end{bmatrix} = \begin{bmatrix} 1 \\ 4 \end{bmatrix}$$

**2. Solve the KKT Subproblem:**
Since there are no active inequality constraints, our KKT matrix only includes $\mathbf{H}$ and $\mathbf{C}_{eq}$.


$$\begin{bmatrix} \mathbf{H} & \mathbf{C}_{eq}^T \\ \mathbf{C}_{eq} & \mathbf{0} \end{bmatrix} \begin{bmatrix} \tilde{\mathbf{s}}^{(0)} \\ \mu^{(0)} \end{bmatrix} = \begin{bmatrix} -\mathbf{h}^{(0)} \\ \mathbf{0} \end{bmatrix}$$

Plugging in the numbers:


$$\begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 0 \\ 1 & 0 & 0 \end{bmatrix} \begin{bmatrix} \tilde{s}_1 \\ \tilde{s}_2 \\ \mu \end{bmatrix} = \begin{bmatrix} -1 \\ -4 \\ 0 \end{bmatrix}$$

Solving this system of linear equations:

* Row 3: $1\tilde{s}_1 = 0 \implies \tilde{s}_1 = 0$
* Row 2: $1\tilde{s}_2 = -4 \implies \tilde{s}_2 = -4$
* Row 1: $1\tilde{s}_1 + 1\mu = -1 \implies 0 + \mu = -1 \implies \mu = -1$

Our optimal step direction is $\tilde{\mathbf{s}}^{(0)} = \begin{bmatrix} 0 \\ -4 \end{bmatrix}$.

**3. Compute Step Size (Line Search):**
We have a non-zero step. We must check if taking a full step ($\alpha = 1$) will violate our inactive inequality constraint.

* Check direction: $\mathbf{c}_i^T \tilde{\mathbf{s}}^{(0)} = \begin{bmatrix} 1 & -0.5 \end{bmatrix} \begin{bmatrix} 0 \\ -4 \end{bmatrix} = 2$.
* Because $2 > 0$, we are walking directly towards the boundary wall.
* Calculate collision fraction: $\alpha_0 = \frac{-d_{ineq} - \mathbf{c}_i^T \mathbf{p}^{(0)}}{\mathbf{c}_i^T \tilde{\mathbf{s}}^{(0)}} = \frac{0 - (1 - 2)}{2} = \frac{1}{2} = 0.5$.

Since $0.5 < 1$, we cannot take the full step. We step exactly onto the boundary.

**4. Update State:**

* $\mathbf{p}^{(1)} = \mathbf{p}^{(0)} + \alpha_0 \tilde{\mathbf{s}}^{(0)} = \begin{bmatrix} 1 \\ 4 \end{bmatrix} + 0.5 \begin{bmatrix} 0 \\ -4 \end{bmatrix} = \begin{bmatrix} 1 \\ 2 \end{bmatrix}$.
* The inequality constraint is now active, so $\mathcal{A}^{(1)} = \{ \text{Constraint } 1 \}$.

---

### Iteration 2: Reaching the Optimum

**1. Calculate the Gradient:**


$$\mathbf{h}^{(1)} = \mathbf{H}\mathbf{p}^{(1)} + \mathbf{g} = \begin{bmatrix} 1 \\ 2 \end{bmatrix}$$

**2. Solve the KKT Subproblem:**
Now, both the equality constraint and the active inequality constraint must be included in the block matrix.


$$\begin{bmatrix} \mathbf{H} & \mathbf{C}_{\mathcal{A}}^T & \mathbf{C}_{eq}^T \\ \mathbf{C}_{\mathcal{A}} & 0 & 0 \\ \mathbf{C}_{eq} & 0 & 0 \end{bmatrix} \begin{bmatrix} \tilde{\mathbf{s}}^{(1)} \\ \lambda^{(1)} \\ \mu^{(1)} \end{bmatrix} = \begin{bmatrix} -\mathbf{h}^{(1)} \\ \mathbf{0} \\ \mathbf{0} \end{bmatrix}$$

Plugging in the numbers:


$$\begin{bmatrix} 1 & 0 & 1 & 1 \\ 0 & 1 & -0.5 & 0 \\ 1 & -0.5 & 0 & 0 \\ 1 & 0 & 0 & 0 \end{bmatrix} \begin{bmatrix} \tilde{s}_1 \\ \tilde{s}_2 \\ \lambda \\ \mu \end{bmatrix} = \begin{bmatrix} -1 \\ -2 \\ 0 \\ 0 \end{bmatrix}$$

Solving the system:

* Row 4: $1\tilde{s}_1 = 0 \implies \tilde{s}_1 = 0$
* Row 3: $1\tilde{s}_1 - 0.5\tilde{s}_2 = 0 \implies 0 - 0.5\tilde{s}_2 = 0 \implies \tilde{s}_2 = 0$

Our step is $\tilde{\mathbf{s}}^{(1)} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$. *(We are in a corner; we cannot move without violating a constraint).*

Now we must solve for the multipliers to see if we are at the global minimum or just trapped.

* Row 2: $1\tilde{s}_2 - 0.5\lambda = -2 \implies 0 - 0.5\lambda = -2 \implies \mathbf{\lambda = 4}$
* Row 1: $1\tilde{s}_1 + 1\lambda + 1\mu = -1 \implies 0 + 4 + \mu = -1 \implies \mathbf{\mu = -5}$

*(Notice that these multiplier values match the assertions exactly in your Image 4 test cases!)*

**3. The Termination Check:**

* Our step $\tilde{\mathbf{s}}^{(1)}$ is zero.
* We check the Lagrange multipliers for the active inequality constraints (Dual Feasibility).
* $\lambda = 4$. Since $4 \ge 0$, the multiplier is positive. This physically means that moving away from this constraint boundary would *increase* the objective function. We are exactly where we want to be.

### Conclusion

The algorithm halts. The optimal point is $\mathbf{p}^* = \begin{bmatrix} 1 \\ 2 \end{bmatrix}$, and it was reached in 2 iterations, perfectly mirroring the behavior of the Python code.

In the cell below, we can see if you successfully disallow infeasible points:

In [10]:
# same setup as before:
H = 1.0 * np.eye(2)
g = np.zeros(2)

C_ineq = np.array([[1.0, -0.5]])
d_ineq = np.array([0.0])

C_eq = np.array([[1.0, 0.0]])
d_eq = np.array([-1.0])


# --- First Feasibility Test ---
p0 = np.array([2.0, 1.0])

exception_thrown = False
try:
    solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0, max_iter=10)
except ValueError:
    exception_thrown = True
    
# Equivalent to Test.@test_throws Exception ...
assert exception_thrown, "Expected a ValueError for infeasible p0 [2, 1]"


# ### BEGIN TESTS
# --- Second Feasibility Test ---
p0 = np.array([0.5, 2.0])

exception_thrown = False
try:
    solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0, max_iter=10)
except ValueError:
    exception_thrown = True
    
# Equivalent to Test.@test_throws Exception ...
assert exception_thrown, "Expected a ValueError for infeasible p0 [0.5, 2]"
# ### END TESTS

print("All infeasibility test blocks passed successfully!")

All infeasibility test blocks passed successfully!


## JuMP for Linear and Quadratic Programming

The [`JuMP` package](https://jump.dev/JuMP.jl/stable/) provides a unified syntax to
define problems with linear or quadratic constraints.
It also supports automatic re-formulation of problems to suit the needs of
the various solvers that are supported.
[The list of solvers](https://jump.dev/JuMP.jl/stable/installation/#Supported-solvers) is
pretty extensive, and the ability to switch between them so easily is one of the
main reasons why `JuMP` is very popular with practitioners in the fields of optimization
and optimal control.

If we were to write an actual SQP algorithm, and did not want to bother with
the Active-Set-Method subroutine ourselves, we could use `JuMP` to solve the sub-problems.
In exercise 2a) you are going to use `JuMP` and the solver `COSMO` to solve the following
problem:
$$
\begin{alignedat}{3}
&\min_{\mathbf p\in ℝ^2}
   (p_1 - 1)^2 + (p_2 - 2.5)^2
&& &&\text{subject to}\\
&-p_1 + 2 p_2 - 2 &&≤ 0,\\
&+p_1 + 2 p_2 - 6 &&≤ 0,\\
&+p_1 - 2 p_2 - 2 &&≤ 0,\\
&+p_1 && \geq 0,\\
&+p_2 && \geq 0.\\
\end{alignedat}
\tag{P}
$$

![problem plot](fig_cosmo.png)

### 1. The Objective Function (The "Bowl")

The goal of the problem is to minimize the function:


$$\min_{\mathbf{p} \in \mathbb{R}^2} (p_1 - 1)^2 + (p_2 - 2.5)^2$$

Geometrically, this equation defines a 3D paraboloid (like a bowl) opening upwards.

* **The Target:** If there were no constraints, the absolute minimum of this function would be at the bottom of the bowl, which is exactly at the coordinates $(1, 2.5)$.
* **The Colors:** In the contour plot, the objective function is represented by the colored bands. Dark purple represents the lowest values (the bottom of the bowl), and the yellow/green colors represent higher values (moving up the sides of the bowl).

### 2. The Linear Constraints (The "Walls")

You cannot simply pick $(1, 2.5)$ because it violates the rules of the system. The linear inequalities in the formulation act as "walls" blocking off parts of the coordinate plane.

Let's map the equations to the solid colored regions in the plot (where $p_1$ is the x-axis and $p_2$ is the y-axis):

* **The Red Region:** corresponds to $-p_1 + 2p_2 - 2 \le 0$. If you rearrange this to solve for $p_2$, you get $p_2 \le 0.5p_1 + 1$. The red area is everything *above* this line, which is forbidden territory. Notice that the unconstrained minimum $(1, 2.5)$ sits squarely inside this illegal red zone.
* **The Orange Region:** corresponds to $p_1 + 2p_2 - 6 \le 0$, or $p_2 \le -0.5p_1 + 3$. The orange area blocks off the top-right corner.
* **The Cyan (Light Blue) Region:** corresponds to $p_1 - 2p_2 - 2 \le 0$, or $p_2 \ge 0.5p_1 - 1$. This blocks off the bottom-right corner.
* **The Bounds:** The final two constraints ($p_1 \ge 0$ and $p_2 \ge 0$) just mean the entire problem is restricted to the upper-right quadrant, which is why the graph's axes start at zero.

### 3. The Feasible Region and the Solution

The area where you can clearly see the contour lines (the polygon bounded by the red, orange, and cyan walls) is the **feasible region**. This represents all the possible combinations of $p_1$ and $p_2$ that satisfy every single constraint.

When you pass this problem to a solver like JuMP using COSMO, the algorithm's job is to find the darkest purple spot that is still inside the feasible region. Because the true minimum is trapped in the red zone, the solver will "slide" down the contour bowl until it bumps into the red wall.

Visually, the optimal solution to this specific QP will lie exactly on the boundary of the red region, at the point where the contour lines are closest to the unconstrained center of $(1, 2.5)$.

### Exercise 2a) 
Below is a function to solve the problem (P) using `JuMP`.
However, there are 5 mistakes in the code.
Fix them. If unsure, consult the [Tutorial.](https://jump.dev/JuMP.jl/stable/tutorials/getting_started/getting_started_with_JuMP/#Getting-started-with-JuMP)

In [11]:
import cvxpy as cp
import numpy as np

In [16]:
def solve_problem_p():
    # 1 & 2) FIXED: Define exactly 2 variables with a lower bound of 0
    p = cp.Variable(2, nonneg=True) 

    # 3) FIXED: Objective function matches the exact 2D paraboloid
    # Note: cvxpy uses 0-based indexing (p[0] is p1, p[1] is p2)
    objective_expr = (p[0] - 1)**2 + (p[1] - 2.5)**2
    objective = cp.Minimize(objective_expr)

    # 4 & 5) FIXED: Correct signs and right-hand side bounds
    constraints = [
        -p[0] + 2 * p[1] - 2 <= 0,
         p[0] + 2 * p[1] - 6 <= 0,
         p[0] - 2 * p[1] - 2 <= 0
    ]

    # Formulate and solve the problem
    # CVXPY will automatically select an appropriate solver (like OSQP)
    model = cp.Problem(objective, constraints)
    model.solve()

    return p.value, model

# Run the solver
p_cosmo, model = solve_problem_p()

print("Optimal Solution (p_cosmo):")
print(f"p1 = {p_cosmo[0]:.4f}")
print(f"p2 = {p_cosmo[1]:.4f}")

Optimal Solution (p_cosmo):
p1 = 1.4000
p2 = 1.7000


Check success:

In [17]:
# --- Check success ---
assert model.status == cp.OPTIMAL, f"Expected OPTIMAL, got {model.status}"

Check dimensions and constraint types:

In [20]:
# --- Check dimensions and constraint types ---
assert len(p_cosmo) == 2, f"Expected length 2, got {len(p_cosmo)}"

# In CVXPY, `nonneg=True` acts on the 2-element vector (2 bounds), 
# plus the 3 explicit inequalities we added. 
# We assert the 3 explicit constraints here to conceptually match JuMP's total of 5.
assert len(model.constraints) == 3, "Expected 3 explicit linear constraints"

Check optimal value:

In [22]:
# --- Check optimal value ---
p = p_cosmo

# Using a tiny tolerance for the >= 0 check due to floating-point solver precision
assert np.all(p >= -1e-10), "Expected all elements to be >= 0"

# Evaluate constraints (0-based indexing)
assert -p[0] + 2 * p[1] - 2 <= 1e-10, "Constraint 1 violated"
assert  p[0] + 2 * p[1] - 6 <= 1e-10, "Constraint 2 violated"
assert  p[0] - 2 * p[1] - 2 <= 1e-10, "Constraint 3 violated"

# ### BEGIN TESTS
# Check if the solution is approximately [1.4, 1.7]
assert np.allclose(p, [1.4, 1.7], atol=1e-4), f"Expected solution ≈ [1.4, 1.7], got {p}"
# ### END TESTS

print("All CVXPY tests passed successfully!")

All CVXPY tests passed successfully!


![problem plot](fig_cosmo2.png)

### Exercise 2b) 
As indicated, it is possible to use `JuMP` in many different ways.
We could also provide the affine-linear inequality constraints with matrices.
Set the 3 x 2 matrix `A_cosmo` and the 3-vector `b_cosmo`, so that `A_cosmo * x .<= b_cosmo`
describes the constraints of (P) (other than the non-negativity constraints.)


### The Concept: Standard Matrix Form

Optimization solvers (like COSMO, OSQP, or the custom Active-Set solver you built) do not understand algebraic strings like `"(p1 - 1)^2"`. They are essentially linear algebra engines. To use them, you must convert your algebraic equations into standard matrix and vector inputs.

This exercise is to map the specific Problem (P) into the universal standard form for Quadratic Programming.

#### 1. Formulating the Constraints ($\mathbf{A}\mathbf{p} \le \mathbf{b}$)

The problem previously defined three linear inequality constraints in the form $c_i(p) \le 0$:

1. $-p_1 + 2p_2 - 2 \le 0$
2. $p_1 + 2p_2 - 6 \le 0$
3. $p_1 - 2p_2 - 2 \le 0$

To get this into the standard $\mathbf{A}\mathbf{p} \le \mathbf{b}$ format, we move the constants to the right-hand side:

1. $-1p_1 + 2p_2 \le 2$
2. $1p_1 + 2p_2 \le 6$
3. $1p_1 - 2p_2 \le 2$

Extracting the coefficients into a matrix $\mathbf{A}$ and the constants into a vector $\mathbf{b}$ gives us:


$$\begin{bmatrix} -1 & 2 \\ 1 & 2 \\ 1 & -2 \end{bmatrix} \begin{bmatrix} p_1 \\ p_2 \end{bmatrix} \le \begin{bmatrix} 2 \\ 6 \\ 2 \end{bmatrix}$$

#### 2. Formulating the Objective Function ($\frac{1}{2}\mathbf{p}^T\mathbf{H}\mathbf{p} + \mathbf{g}^T\mathbf{p} + c$)

The objective is to minimize $f(\mathbf{p}) = (p_1 - 1)^2 + (p_2 - 2.5)^2$.
First, expand the binomials:


$$f(\mathbf{p}) = (p_1^2 - 2p_1 + 1) + (p_2^2 - 5p_2 + 6.25)$$

$$f(\mathbf{p}) = p_1^2 + p_2^2 - 2p_1 - 5p_2 + 7.25$$

Now, map these terms to the standard quadratic formulation:

* **The Quadratic Term ($\mathbf{H}$):** We need a matrix $\mathbf{H}$ such that $\frac{1}{2}\mathbf{p}^T\mathbf{H}\mathbf{p} = p_1^2 + p_2^2$. Because of the $\frac{1}{2}$ multiplier in the standard formula, the diagonal elements of $\mathbf{H}$ must be $2$.

$$\frac{1}{2} \begin{bmatrix} p_1 & p_2 \end{bmatrix} \begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix} \begin{bmatrix} p_1 \\ p_2 \end{bmatrix} = p_1^2 + p_2^2$$


* **The Linear Term ($\mathbf{g}$):** We need a vector $\mathbf{g}$ such that $\mathbf{g}^T\mathbf{p} = -2p_1 - 5p_2$.

$$\begin{bmatrix} -2 & -5 \end{bmatrix} \begin{bmatrix} p_1 \\ p_2 \end{bmatrix} = -2p_1 - 5p_2$$


* **The Constant ($c$):** The leftover scalar values added together ($1 + 6.25 = 7.25$).

In [26]:
# --- Part 1: Affine-linear inequality constraints ---
# A_cosmo * x <= b_cosmo

A_cosmo = np.zeros((3, 2))
b_cosmo = np.zeros(3)

### BEGIN SOLUTION
A_cosmo = np.array([
    [-1.0,  2.0],
    [ 1.0,  2.0],
    [ 1.0, -2.0]
])

b_cosmo = np.array([2.0, 6.0, 2.0])
### END SOLUTION

Moreover, we can also rewrite the objective function as
`0.5 * p' * H_cosmo * p + p'g_cosmo + c_cosmo`,
with a 2 x 2 matrix `H_cosmo` and vector `g_cosmo` and constant `c_cosmo`:

In [23]:
# --- Part 2: Objective function ---
# 0.5 * p' * H_cosmo * p + p' * g_cosmo + c_cosmo

H_cosmo = np.zeros((2, 2))
g_cosmo = np.zeros(2)
c_cosmo = 0.0

### BEGIN SOLUTION
# Note: Diagonals are 2 to cancel out the 0.5 multiplier in the formulation
H_cosmo = np.array([
    [2.0, 0.0],
    [0.0, 2.0]
])

g_cosmo = np.array([-2.0, -5.0])

c_cosmo = 7.25
### END SOLUTION

print("Matrix formulations successfully defined.")

Matrix formulations successfully defined.


Check against last optimal value:

In [24]:
# Retrieve the optimal objective value calculated by the CVXPY solver
solver_obj_value = model.value

# Calculate the objective value manually using our matrix formulation
manual_obj_value = 0.5 * p_cosmo.T @ H_cosmo @ p_cosmo + p_cosmo.T @ g_cosmo + c_cosmo

assert np.isclose(solver_obj_value, manual_obj_value), \
    f"Mismatch! Solver value: {solver_obj_value}, Manual value: {manual_obj_value}"

print("Objective value correctly matches the matrix formulation!")

Objective value correctly matches the matrix formulation!


Here is how to use matrices with `JuMP`:

In [ ]:
# 1. Define the variable with non-negativity bounds (p >= 0)
p = cp.Variable(2, nonneg=True)

# 2. Define the objective function using matrices
# We use cp.quad_form(p, H_cosmo) to satisfy CVXPY's strict convexity rules
objective_expr = 0.5 * cp.quad_form(p, H_cosmo) + g_cosmo.T @ p + c_cosmo
objective = cp.Minimize(objective_expr)

# 3. Define the constraints using matrices
constraints = [A_cosmo @ p <= b_cosmo]

# 4. Initialize and solve the model
model = cp.Problem(objective, constraints)
model.solve()

# 5. Extract the optimal values
p_opt = p.value

# 6. Assert that the matrix formulation yields the same result
assert np.allclose(p_opt, p_cosmo, atol=1e-5), \
    f"Mismatch! Matrix formulation: {p_opt}, Expected: {p_cosmo}"

print("Matrix-based optimization successful!")

### Explanation of the Concepts

This snippet demonstrates how to formulate and solve a **Quadratic Program (QP)** using standard matrix notation instead of writing out the individual algebraic terms. This is how large-scale optimization problems (like those running inside Model Predictive Control algorithms) are handled in practice.

Here is a step-by-step breakdown of what the code is doing:

* **Variable Declaration (`@variable` / `cp.Variable`):** It creates a 2-dimensional vector of decision variables, $\mathbf{p} = \begin{bmatrix} p_1 \\ p_2 \end{bmatrix}$. The `>= 0` condition establishes the baseline bounds, ensuring both variables remain non-negative.
* **The Objective Function (`@objective` / `cp.Minimize`):** It defines the cost function to be minimized using standard quadratic matrix form:

$$L(\mathbf{p}) = \frac{1}{2}\mathbf{p}^T\mathbf{H}\mathbf{p} + \mathbf{p}^T\mathbf{g} + c$$



Instead of writing out $(p_1 - 1)^2 + (p_2 - 2.5)^2$, the solver relies on the pre-constructed Hessian matrix ($\mathbf{H}$), gradient vector ($\mathbf{g}$), and scalar constant ($c$) to map the curvature and position of the 3D "bowl" we are trying to find the bottom of. *Note: In CVXPY, `cp.quad_form` must be used instead of standard matrix multiplication to prove to the solver that the quadratic equation is convex.*
* **The Constraints (`@constraint` / `constraints = [...]`):** It applies all the linear inequality "walls" at once using the matrix equation:

$$\mathbf{A}\mathbf{p} \le \mathbf{b}$$



By multiplying the coefficient matrix $\mathbf{A}$ with our variable vector $\mathbf{p}$, it evaluates all constraints simultaneously.
* **Solving and Verification (`optimize!` / `model.solve()`):** It hands the matrices to the underlying mathematical solver (like COSMO or OSQP). Once a solution is found, it extracts the optimal point (`p_opt`) and uses an assertion to verify that this matrix-based approach found the exact same geometric coordinate as the algebraic approach (`p_cosmo`).

### Exercise 2c) 
Incidently, we have restated our problem in a form nearly suitable for our Active-Set solver
from exercise 1.
Augment `A_cosmo` and `b_cosmo` by including the non-negativity constraints and
name the resulting matrix `C_ineq_as`, and the vector `b_ineq_as`:

In [27]:
# --- Part 1: Augmenting the Inequality Matrices ---
# We need to add the non-negativity constraints: p1 >= 0 and p2 >= 0
# In <= form, this is: -p1 <= 0 and -p2 <= 0

# Create the matrix for the new bounds
non_neg_A = np.array([
    [-1.0,  0.0],
    [ 0.0, -1.0]
])
non_neg_b = np.array([0.0, 0.0])

### BEGIN SOLUTION
# Stack the old 3x2 matrix with the new 2x2 matrix to get a 5x2 matrix
C_ineq_as = np.vstack((A_cosmo, non_neg_A))

# Concatenate the old length-3 vector with the new length-2 vector
b_ineq_as = np.concatenate((b_cosmo, non_neg_b))
### END SOLUTION

Finally, our solver should return the same optimum, hopefully:

In [ ]:
# --- Part 2: Running the Custom Active-Set Solver ---
# Map the COSMO standard forms to our custom solver's expected inputs
H = H_cosmo
g = g_cosmo
C_ineq = C_ineq_as

# Note: Our solver expects C*p + d <= 0, but COSMO expects A*p <= b. 
# Therefore, A*p - b <= 0, meaning d = -b.
d_ineq = -b_ineq_as 

# No equality constraints in this problem
C_eq = np.zeros((0, 2))
d_eq = np.zeros(0)

# Initial guess (p0 = ones(2))
p0 = np.ones(2)

# Run the custom solver we built in Exercise 1
p_qp, iterations = solve_convex_qp(H, g, C_eq, d_eq, C_ineq, d_ineq, p0)

# Assert   our custom hand-built solver matches the professional JuMP/CVXPY solver
assert np.allclose(p_cosmo, p_qp, atol=1e-5), \
    f"Mismatch! JuMP/CVXPY optimum: {p_cosmo}, Custom Solver optimum: {p_qp}"

print("Success! The custom Active-Set solver found the exact same optimum.")
print(f"Optimal Point: {p_qp}")

Iteration 1, L(pk)=-5.0
Iteration 2, L(pk)=-6.25
Iteration 3, L(pk)=-6.450000000000001
Zero step.
Problem solved.


Success! The custom Active-Set solver found the exact same optimum.
Optimal Point: [1.4 1.7]
